In [58]:
import sqlite3

paths = [
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq128_bs4_fp32.sqlite",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq512_bs4_fp32.sqlite",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq4096_bs4_fp32.sqlite",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq128_bs4_fp32.sqlite",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq512_bs4_fp32.sqlite",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq1024_bs4_fp32.sqlite"
]

config_path_mapping = {
    path.split('/')[-1].split(".sqlite")[0]: path for path in paths
}


In [59]:
config_path_mapping

{'small_seq128_bs4_fp32': '../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq128_bs4_fp32.sqlite',
 'small_seq512_bs4_fp32': '../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq512_bs4_fp32.sqlite',
 'small_seq4096_bs4_fp32': '../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq4096_bs4_fp32.sqlite',
 'xl_seq128_bs4_fp32': '../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq128_bs4_fp32.sqlite',
 'xl_seq512_bs4_fp32': '../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq512_bs4_fp32.sqlite',
 'xl_seq1024_bs4_fp32': '../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq1024_bs4_fp32.sqlite'}

In [63]:
import statistics

results_df_rows = []

for config, path in config_path_mapping.items():
    connection = sqlite3.connect(path)
    cursor = connection.cursor()

    forward_sql_code = """
    SELECT
        COALESCE(n.text, s.value) AS name,
        n.start,
        n.end
    FROM NVTX_EVENTS AS n
    LEFT JOIN StringIds AS s ON n.textId = s.id
    WHERE COALESCE(n.text, s.value) LIKE 'step_%/forward'
    ORDER BY n.start;
    """

    cursor.execute(forward_sql_code)
    result = cursor.fetchall()

    cpu_forward_times_ms = []

    for res in result:
        cpu_forward_times_ms.append((res[2] - res[1]) / 10 ** 6)

    print(f"time for forward: {statistics.mean(cpu_forward_times_ms)} +- {statistics.stdev(cpu_forward_times_ms)} ms")

    backward_sql_code = """
    SELECT
        COALESCE(n.text, s.value) AS name,
        n.start,
        n.end
    FROM NVTX_EVENTS AS n
    LEFT JOIN StringIds AS s ON n.textId = s.id
    WHERE COALESCE(n.text, s.value) LIKE 'torch.autograd.backward'
    ORDER BY n.start;
    """

    cursor.execute(backward_sql_code)
    result = cursor.fetchall()
    
    cpu_backward_times_ms = []

    for res in result:
        cpu_backward_times_ms.append((res[2] - res[1]) / 10 ** 6)

    print(f"time for backward: {statistics.mean(cpu_backward_times_ms)} +- {statistics.stdev(cpu_backward_times_ms)} ms")

    cpu_fb_times_ms = []

    for forward_time, backward_time in zip(cpu_forward_times_ms, cpu_backward_times_ms):
        cpu_fb_times_ms.append(forward_time + backward_time)

    print(f"time for forward + backward: {statistics.mean(cpu_fb_times_ms)} +- {statistics.stdev(cpu_fb_times_ms)} ms")

    optimizer_sql_code = """
    SELECT
        COALESCE(n.text, s.value) AS name,
        n.start,
        n.end
    FROM NVTX_EVENTS AS n
    LEFT JOIN StringIds AS s ON n.textId = s.id
    WHERE COALESCE(n.text, s.value) LIKE 'AdamW.step'
    ORDER BY n.start;
    """

    cursor.execute(optimizer_sql_code)
    result = cursor.fetchall()

    cpu_optimizer_times_ms = []

    for res in result:
        cpu_optimizer_times_ms.append((res[2] - res[1]) / 10 ** 6)

    print(f"time for optimizer: {statistics.mean(cpu_optimizer_times_ms)} +- {statistics.stdev(cpu_optimizer_times_ms)} ms")

    cpu_fbo_times_ms = []

    for forward_time, backward_time, optimizer_time in zip(cpu_forward_times_ms, cpu_backward_times_ms, cpu_optimizer_times_ms):
        cpu_fbo_times_ms.append(forward_time + backward_time + optimizer_time)

    print(f"time for forward + backward + optimizer: {statistics.mean(cpu_fbo_times_ms)} +- {statistics.stdev(cpu_fbo_times_ms)} ms")

    result_df_row = {
        "config": config,
        "forward_times_mean": statistics.mean(cpu_forward_times_ms),
        "forward_times_std": statistics.stdev(cpu_forward_times_ms),
        "backward_times_mean": statistics.mean(cpu_backward_times_ms),
        "backward_times_std": statistics.stdev(cpu_backward_times_ms),
        "optimizer_times_mean": statistics.mean(cpu_optimizer_times_ms),
        "optimizer_times_std": statistics.stdev(cpu_optimizer_times_ms),
        "forwad+backward_times_mean": statistics.mean(cpu_fb_times_ms),
        "forwad+backward_times_std": statistics.stdev(cpu_fb_times_ms),
        "forward+backward+optimizer_times_mean": statistics.mean(cpu_fbo_times_ms),
        "forward+backward+optimizer_times_std": statistics.stdev(cpu_fbo_times_ms),
    }

    results_df_rows.append(result_df_row)

time for forward: 31.0429569 +- 12.089597007699155 ms
time for backward: 51.9928441 +- 26.62416685410959 ms
time for forward + backward: 83.03580099999999 +- 29.51705663172357 ms
time for optimizer: 22.9508702 +- 12.563189602491734 ms
time for forward + backward + optimizer: 105.9866712 +- 41.38080361158804 ms
time for forward: 31.2574328 +- 4.369274863807973 ms
time for backward: 53.863613199999996 +- 22.71498409598575 ms
time for forward + backward: 85.12104599999999 +- 23.171400421599735 ms
time for optimizer: 20.8335266 +- 3.115037963336213 ms
time for forward + backward + optimizer: 105.95457259999999 +- 25.838086582158994 ms
time for forward: 264.9648831 +- 6.064642276465643 ms
time for backward: 285.5206424 +- 104.86958558867666 ms
time for forward + backward: 550.4855255 +- 110.85142250126346 ms
time for optimizer: 282.897028 +- 90.19706620247298 ms
time for forward + backward + optimizer: 833.3825535000001 +- 20.666347145011223 ms
time for forward: 143.5578278 +- 3.35098516706

In [66]:
import pandas as pd

results_df = pd.DataFrame(results_df_rows)
results_df.head()

results_df.to_csv("../benches/nsys_runs/assignment_runs.csv", index=False)

In [69]:
results_df

,config,forward_times_mean,forward_times_std,backward_times_mean,backward_times_std,optimizer_times_mean,optimizer_times_std,forwad+backward_times_mean,forwad+backward_times_std,forward+backward+optimizer_times_mean,forward+backward+optimizer_times_std
0,small_seq128_bs4_fp32,31.042957,12.089597,51.992844,26.624167,22.950870,12.563190,83.035801,29.517057,105.986671,41.380804
1,small_seq512_bs4_fp32,31.257433,4.369275,53.863613,22.714984,20.833527,3.115038,85.121046,23.171400,105.954573,25.838087
2,small_seq4096_bs4_fp32,264.964883,6.064642,285.520642,104.869586,282.897028,90.197066,550.485525,110.851423,833.382554,20.666347
3,xl_seq128_bs4_fp32,143.557828,3.350985,166.067098,55.405439,87.987998,19.107851,309.624926,54.042296,397.612923,71.144687
4,xl_seq512_bs4_fp32,317.691778,16.451605,456.718708,58.449301,188.869304,12.007225,774.410486,58.561036,963.279790,47.124305
5,xl_seq1024_bs4_fp32,602.919144,12.381229,939.449318,100.542163,302.117571,79.811720,1542.368462,101.267694,1844.486033,23.933857
